# Autoencoder Similarity Model

Model 3 of 3. This notebook learns a dense embedding for each paper and uses cosine similarity to rank related papers.

**Evaluation:** held-out reconstruction MSE, top-k same-subfield agreement, and a matched-dimension TruncatedSVD baseline.


## Setup

In [ ]:
%pip install torch --quiet
%pip install lightning --quiet


In [ ]:
import json
import sqlite3
import warnings
from pathlib import Path

import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
import torch.nn as nn
from scipy.sparse import csr_matrix, hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import ParameterSampler, train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED = 42
L.seed_everything(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"PyTorch: {torch.__version__} | Lightning: {L.__version__}")


## Load Data

Same OpenAlex ↔ Semantic Scholar join as the K-Means notebook. Keep only rows with an abstract vector and a non-empty TLDR so every paper has the same feature blocks.


In [ ]:
DB_PATH = Path("papers.db")
if not DB_PATH.exists():
    DB_PATH = Path("../papers.db")

JOINED_QUERY = """
SELECT
    o.openalex_id,
    o.doi,
    o.doi_normalized,
    o.title,
    o.publication_year,
    o.cited_by_count,
    o.author_count,
    o.primary_topic,
    o.primary_subfield,
    o.primary_field,
    o.primary_domain,
    s.abstract_text,
    s.abstract_tfidf_vector,
    s.tldr_text
FROM openalex_papers AS o
JOIN semanticscholar_papers AS s
    ON o.doi_normalized = s.doi_normalized
WHERE s.abstract_text IS NOT NULL
  AND TRIM(s.abstract_text) <> ''
  AND s.abstract_tfidf_vector IS NOT NULL
  AND TRIM(s.abstract_tfidf_vector) <> ''
  AND s.tldr_text IS NOT NULL
  AND TRIM(s.tldr_text) <> ''
"""

with sqlite3.connect(DB_PATH) as conn:
    df = pl.read_database(query=JOINED_QUERY, connection=conn)

print(f"Joined rows: {df.shape[0]}")
df.select(["title", "publication_year", "primary_subfield"]).head(5)


## Train / Validation / Test Split

Row-level 70 / 15 / 15 split, matched to the K-Means notebook. Every transformer (TF-IDF, SVD, scaler, autoencoder) is fit on **train only**.


In [ ]:
SPLIT_SEED = 42

train_idx, temp_idx = train_test_split(
    np.arange(df.height), test_size=0.30, random_state=SPLIT_SEED, shuffle=True,
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=SPLIT_SEED, shuffle=True,
)

train_df = df[train_idx]
val_df   = df[val_idx]
test_df  = df[test_idx]

pl.DataFrame({
    "split": ["train", "val", "test"],
    "rows":  [train_df.height, val_df.height, test_df.height],
})


## Feature Engineering — Leakage-Safe

Feature blocks:
- structured: `publication_year`, `log(author_count+1)`
- abstract: stored 2000-dim TF-IDF
- TLDR: `TfidfVectorizer(max_features=750)`, fit on train only

`primary_subfield` and `primary_topic` are dropped because they map too directly to the evaluation target. `primary_field`, `primary_domain`, and `cited_by_count` are also dropped to avoid low-variance or popularity-dominated features.


In [ ]:
# Drop primary_subfield and primary_topic so the same-subfield evaluation
# below isn't effectively given the label. Use the train-set median for missing publication years.
CATEGORICAL_FEATURES = []
YEAR_FILL_VALUE = float(train_df["publication_year"].median())


def build_structured(frame, reference_columns=None):
    """Numeric + one-hot categorical block. Uses reference_columns to align test/val to train."""
    encoded = frame.select([
        pl.col("publication_year").fill_null(YEAR_FILL_VALUE).cast(pl.Float64),
        pl.col("author_count").fill_null(0).clip(lower_bound=0).log1p().alias("log_author_count"),
    ])
    if reference_columns is None:
        return encoded, encoded.columns
    # Align shape: add missing train columns as zeros, drop unseen columns.
    missing = [c for c in reference_columns if c not in encoded.columns]
    if missing:
        encoded = encoded.with_columns([pl.lit(0).alias(c) for c in missing])
    return encoded.select(reference_columns), reference_columns


train_struct_df, struct_columns = build_structured(train_df)
val_struct_df,   _ = build_structured(val_df,  struct_columns)
test_struct_df,  _ = build_structured(test_df, struct_columns)

print(f"Year fill value (train median): {YEAR_FILL_VALUE}")
print(f"Structured columns: {len(struct_columns)}")


In [ ]:
def stored_tfidf_to_csr(vector_jsons):
    """Deserialize the JSON-encoded stored abstract TF-IDF vectors into a single CSR matrix."""
    data, indices, indptr = [], [], [0]
    dimension = None
    for vj in vector_jsons:
        v = json.loads(vj)
        if dimension is None:
            dimension = v["dimension"]
        indices.extend(v["indices"])
        data.extend(v["values"])
        indptr.append(len(indices))
    return csr_matrix((data, indices, indptr), shape=(len(vector_jsons), dimension or 0))


# Abstract block — stored vectors, identical for every split.
abs_train = stored_tfidf_to_csr(train_df["abstract_tfidf_vector"].to_list())
abs_val   = stored_tfidf_to_csr(val_df["abstract_tfidf_vector"].to_list())
abs_test  = stored_tfidf_to_csr(test_df["abstract_tfidf_vector"].to_list())

# TLDR block — vectorizer fit on TRAIN only.
tldr_vectorizer = TfidfVectorizer(stop_words="english", max_features=750, min_df=3)
tldr_train = tldr_vectorizer.fit_transform(train_df["tldr_text"].to_list())
tldr_val   = tldr_vectorizer.transform(val_df["tldr_text"].to_list())
tldr_test  = tldr_vectorizer.transform(test_df["tldr_text"].to_list())

print(f"Abstract TF-IDF shape (train): {abs_train.shape}")
print(f"TLDR     TF-IDF shape (train): {tldr_train.shape}")


In [ ]:
# Scale the structured block only (TF-IDF is already L2-ish; scaling sparse text often hurts).
struct_scaler = StandardScaler(with_mean=False)  # with_mean=False keeps output sparse-friendly
struct_train = csr_matrix(struct_scaler.fit_transform(train_struct_df.to_numpy()))
struct_val   = csr_matrix(struct_scaler.transform(val_struct_df.to_numpy()))
struct_test  = csr_matrix(struct_scaler.transform(test_struct_df.to_numpy()))

X_train_sparse = hstack([struct_train, abs_train, tldr_train], format="csr")
X_val_sparse   = hstack([struct_val,   abs_val,   tldr_val],   format="csr")
X_test_sparse  = hstack([struct_test,  abs_test,  tldr_test],  format="csr")

print(f"Sparse feature matrix (train): {X_train_sparse.shape}")
print(f"Sparse feature matrix (val):   {X_val_sparse.shape}")
print(f"Sparse feature matrix (test):  {X_test_sparse.shape}")


## Dense Input — TruncatedSVD

Project the sparse matrix to 150 dense dimensions with TruncatedSVD, then standardize before training the autoencoder.


In [ ]:
SVD_DIM = 150

pre_svd = TruncatedSVD(n_components=SVD_DIM, random_state=SEED)
X_train_dense = pre_svd.fit_transform(X_train_sparse)
X_val_dense   = pre_svd.transform(X_val_sparse)
X_test_dense  = pre_svd.transform(X_test_sparse)

dense_scaler = StandardScaler()
X_train_dense = dense_scaler.fit_transform(X_train_dense)
X_val_dense   = dense_scaler.transform(X_val_dense)
X_test_dense  = dense_scaler.transform(X_test_dense)

print(f"Dense input shape (train): {X_train_dense.shape}")
print(f"Explained variance at {SVD_DIM} SVD components: {pre_svd.explained_variance_ratio_.sum():.1%}")


## Autoencoder Model

Symmetric MLP autoencoder in PyTorch Lightning. Encoder: 150d → `latent_dim`; decoder reconstructs back to 150d. Loss = MSE.


In [ ]:
class PaperAutoencoder(L.LightningModule):
    """Symmetric MLP autoencoder for dense paper feature vectors."""

    def __init__(self, input_dim: int, latent_dim: int = 32, hidden_dim: int = 96, lr: float = 1e-3):
        super().__init__()
        self.save_hyperparameters()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )
        self.loss_function = nn.MSELoss()

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

    def training_step(self, batch, batch_idx):
        x = batch[0]
        x_hat, _ = self(x)
        loss = self.loss_function(x_hat, x)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x = batch[0]
        x_hat, _ = self(x)
        loss = self.loss_function(x_hat, x)
        self.log("val_loss", loss, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


def make_loader(X, batch_size=128, shuffle=False):
    ds = torch.utils.data.TensorDataset(torch.FloatTensor(X))
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


INPUT_DIM = X_train_dense.shape[1]
train_loader = make_loader(X_train_dense, shuffle=True)
val_loader   = make_loader(X_val_dense)
test_loader  = make_loader(X_test_dense)
print(f"Autoencoder input dim: {INPUT_DIM}")


## Hyperparameter Tuning — Randomized Search

Sample 6 configs over `latent_dim`, `hidden_dim`, and `lr`. Rank them by validation reconstruction MSE.


In [ ]:
HP_EPOCHS = 20   # short run per config during the search
FINAL_EPOCHS = 60  # full training for the winner
HP_BUDGET = 6

param_space = {
    "latent_dim": [16, 32, 64],
    "hidden_dim": [64, 96, 128],
    "lr":         [1e-3, 5e-4, 1e-4],
}
sampler = list(ParameterSampler(param_space, n_iter=HP_BUDGET, random_state=SEED))

def train_and_val(cfg, max_epochs):
    model = PaperAutoencoder(input_dim=INPUT_DIM, **cfg)
    trainer = L.Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
    )
    trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
    # Compute final validation MSE explicitly for ranking.
    model.eval()
    with torch.no_grad():
        val_recon_losses = []
        for (xb,) in val_loader:
            xhat, _ = model(xb)
            val_recon_losses.append(float(((xhat - xb) ** 2).mean()))
    return model, float(np.mean(val_recon_losses))

hp_results = []
for cfg in sampler:
    _, val_mse = train_and_val(cfg, HP_EPOCHS)
    hp_results.append({**cfg, "val_mse": round(val_mse, 5)})
    print(f"  {cfg}  -> val MSE = {val_mse:.5f}")

hp_df = pl.DataFrame(hp_results).sort("val_mse")
best_cfg = {k: hp_df.row(0, named=True)[k] for k in ["latent_dim", "hidden_dim", "lr"]}
print(f"\nBest config: {best_cfg}  (val MSE = {hp_df.row(0, named=True)['val_mse']})")
hp_df


## Train the Final Autoencoder

Retrain the best config for more epochs and track train/validation loss.


In [ ]:
train_losses, val_losses = [], []

class LossTracker(L.Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        train_losses.append(float(trainer.callback_metrics.get("train_loss", float("nan"))))
    def on_validation_epoch_end(self, trainer, pl_module):
        val_losses.append(float(trainer.callback_metrics.get("val_loss", float("nan"))))

final_model = PaperAutoencoder(input_dim=INPUT_DIM, **best_cfg)
final_trainer = L.Trainer(
    max_epochs=FINAL_EPOCHS,
    accelerator="auto",
    enable_progress_bar=False,
    enable_model_summary=False,
    logger=False,
    callbacks=[LossTracker()],
)
final_trainer.fit(final_model, train_dataloaders=train_loader, val_dataloaders=val_loader)

print(f"Trained {FINAL_EPOCHS} epochs. Final train loss: {train_losses[-1]:.5f}, val loss: {val_losses[-1]:.5f}")


## Extract Embeddings

Encode each split into its final latent vectors. These embeddings drive the similarity search.


In [ ]:
def encode(model, X):
    model.eval()
    with torch.no_grad():
        _, z = model(torch.FloatTensor(X))
    return z.numpy()

ae_train_emb = encode(final_model, X_train_dense)
ae_val_emb   = encode(final_model, X_val_dense)
ae_test_emb  = encode(final_model, X_test_dense)

print(f"Autoencoder embedding shape (test): {ae_test_emb.shape}")


## Similarity Lookup

Rank nearest papers by cosine similarity in the learned embedding space.


In [ ]:
PAPER_LOOKUP_COLUMNS = ["title", "doi", "doi_normalized", "openalex_id"]

def find_paper_index(paper, papers_df):
    """Resolve a paper reference to a row index in papers_df."""
    if isinstance(paper, int):
        if 0 <= paper < papers_df.height:
            return paper
        raise ValueError(f"Row index {paper} out of range 0..{papers_df.height - 1}")

    query = str(paper).strip().lower()
    if not query:
        raise ValueError("Empty paper reference.")

    records = papers_df.select(PAPER_LOOKUP_COLUMNS).to_dicts()
    # Exact match on identifier or title.
    for idx, rec in enumerate(records):
        for col in PAPER_LOOKUP_COLUMNS:
            val = rec.get(col)
            if val is not None and str(val).strip().lower() == query:
                return idx
    # Substring fallback.
    matches = [
        i for i, r in enumerate(records)
        if r.get("title") and query in str(r["title"]).lower()
    ]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        ambig = papers_df[matches].select(["title", "doi_normalized", "publication_year"])
        raise ValueError(f"{len(matches)} title matches; disambiguate with a DOI or row index:\n{ambig}")
    raise ValueError(f"No paper found for: {paper}")


def find_similar_papers(paper, n=10, include_query=False,
                         embeddings=ae_test_emb, papers_df=test_df):
    """Rank papers by cosine similarity to a query paper in the autoencoder embedding space."""
    query_idx = find_paper_index(paper, papers_df)
    query_vec = embeddings[query_idx : query_idx + 1]

    sims = cosine_similarity(query_vec, embeddings).ravel()
    order = np.argsort(-sims)  # descending

    if not include_query:
        order = [i for i in order if i != query_idx]
    top = order[:n]
    return (
        papers_df[list(top)]
        .with_columns(pl.Series("similarity", [float(sims[i]) for i in top]))
        .select(["similarity", "title", "publication_year", "primary_topic", "primary_subfield", "doi"])
    )


# Demo query — the same pattern as the K-Means notebook.
example_title = test_df[0, "title"]
print(f"Query paper: {example_title}\n")
find_similar_papers(example_title, n=5)


## Evaluation

### 1. Reconstruction MSE
Does the latent embedding preserve the SVD input? Low held-out MSE = good compression.


In [ ]:
def reconstruction_mse(model, X):
    model.eval()
    with torch.no_grad():
        xhat, _ = model(torch.FloatTensor(X))
    return float(((xhat.numpy() - X) ** 2).mean())

recon = {
    "train": reconstruction_mse(final_model, X_train_dense),
    "val":   reconstruction_mse(final_model, X_val_dense),
    "test":  reconstruction_mse(final_model, X_test_dense),
}
pl.DataFrame({"split": list(recon), "reconstruction_mse": [round(v, 5) for v in recon.values()]})


### 2. Top-k Same-Subfield Agreement

For each test paper, measure what fraction of its top-k neighbors share its `primary_subfield`. Higher is better.

Because `primary_subfield` and `primary_topic` were dropped from the inputs, this checks whether the embedding recovered subfield structure rather than memorizing the label.


In [ ]:
def topk_same_label_agreement(embeddings, labels, k=5):
    """Mean fraction of the top-k neighbors that share the query's label."""
    sims = cosine_similarity(embeddings)
    np.fill_diagonal(sims, -np.inf)  # exclude self
    top_idx = np.argpartition(-sims, kth=k, axis=1)[:, :k]  # top-k per row (unordered)
    labels_arr = np.asarray(labels)
    neighbor_labels = labels_arr[top_idx]
    return float((neighbor_labels == labels_arr[:, None]).mean())


test_subfield = test_df["primary_subfield"].fill_null("Unknown").to_list()
ae_top5  = topk_same_label_agreement(ae_test_emb, test_subfield, k=5)
ae_top10 = topk_same_label_agreement(ae_test_emb, test_subfield, k=10)

print(f"Autoencoder top-5  same-subfield agreement: {ae_top5:.3f}")
print(f"Autoencoder top-10 same-subfield agreement: {ae_top10:.3f}")


### 3. Baseline Comparison — TruncatedSVD at Matched Dimension

Compare against a TruncatedSVD embedding at the same output dimension to isolate the value of the autoencoder's non-linearity.


In [ ]:
LATENT = int(best_cfg["latent_dim"])

baseline_svd = TruncatedSVD(n_components=LATENT, random_state=SEED)
svd_train_emb = baseline_svd.fit_transform(X_train_sparse)
svd_test_emb  = baseline_svd.transform(X_test_sparse)

# Normalize so cosine-similarity has the same interpretation as for the autoencoder.
svd_test_emb_norm = svd_test_emb / (np.linalg.norm(svd_test_emb, axis=1, keepdims=True) + 1e-9)
ae_test_emb_norm  = ae_test_emb  / (np.linalg.norm(ae_test_emb,  axis=1, keepdims=True) + 1e-9)

svd_top5  = topk_same_label_agreement(svd_test_emb_norm, test_subfield, k=5)
svd_top10 = topk_same_label_agreement(svd_test_emb_norm, test_subfield, k=10)

majority_rate = max(pl.Series(test_subfield).value_counts().get_column("count").to_list()) / len(test_subfield)

comparison_df = pl.DataFrame({
    "embedding":              ["Autoencoder", f"TruncatedSVD @ {LATENT}d", "Majority-class baseline"],
    "top_5_same_subfield":    [round(ae_top5, 3), round(svd_top5, 3), round(majority_rate, 3)],
    "top_10_same_subfield":   [round(ae_top10, 3), round(svd_top10, 3), round(majority_rate, 3)],
})
comparison_df


## Visualizations

In [ ]:
# Training / validation loss curves over epochs.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(train_losses) + 1), train_losses, label="Train", color="steelblue")
# Lightning can log an extra validation_epoch_end pass at step 0 before training starts; align manually.
val_plot = val_losses[-len(train_losses):] if len(val_losses) >= len(train_losses) else val_losses
ax.plot(range(1, len(val_plot) + 1), val_plot, label="Validation", color="darkorange")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Autoencoder Training & Validation Loss")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Bar chart: top-k same-subfield agreement, autoencoder vs TruncatedSVD vs majority baseline.
metrics = ["top_5_same_subfield", "top_10_same_subfield"]
names = comparison_df["embedding"].to_list()
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 4.5))
palette = ["#C44E52", "#4C72B0", "#8c8c8c"]
for i, name in enumerate(names):
    vals = [comparison_df[m][i] for m in metrics]
    ax.bar(x + i * width, vals, width, label=name, color=palette[i])

ax.set_xticks(x + width)
ax.set_xticklabels(["top-5", "top-10"])
ax.set_ylabel("Fraction of neighbors in same subfield")
ax.set_ylim(0, 1)
ax.set_title("Similarity Quality: Same-Subfield Agreement on Test Set")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 2D projection of the learned embedding (PCA over the 32-dim autoencoder output) colored by subfield.
proj = TruncatedSVD(n_components=2, random_state=SEED)
emb_2d = proj.fit_transform(ae_test_emb)

unique_subs = sorted(set(test_subfield))
palette_2d = plt.cm.tab20(np.linspace(0, 1, len(unique_subs)))
sub_to_color = {s: palette_2d[i] for i, s in enumerate(unique_subs)}

fig, ax = plt.subplots(figsize=(10, 6))
for sub in unique_subs:
    mask = np.array([s == sub for s in test_subfield])
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
               s=10, alpha=0.65, color=sub_to_color[sub], label=sub)
ax.set_title("2D Projection of Autoencoder Embedding (Test Set, colored by primary_subfield)")
ax.set_xlabel("Component 1")
ax.set_ylabel("Component 2")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8)
plt.tight_layout()
plt.show()


## Conclusions & Limitations

**What this adds:** a continuous similarity ranking, finer neighborhoods than K-Means, and a non-linear alternative to matched-dimension TruncatedSVD.

**Limitations:** the model optimizes reconstruction, not similarity directly, and the same-subfield metric is still shaped by an imbalanced Computer Science-only dataset.
